In [2]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('../models/bge-m3', use_fp16=True)
model

d:\hybridsearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sentences_1 = [
    "What is BGE M3?",
]

sentences_2 = [
    "BGE M3 is an embedding model supporting dense retrieval, lexical matching, and multi-vector interaction.",
    "BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document.",
    "Doc2Vec extends Word2Vec by learning document-level embeddings alongside word embeddings, capturing document context.",
    "FastText is designed to quickly generate word embeddings by treating each word as composed of character n-grams, improving handling of rare words.",
    "GloVe (Global Vectors for Word Representation) embeddings are generated from aggregated global word-word co-occurrence statistics, emphasizing word relationships.",
    "Siamese networks are used in vector embeddings to learn similarity or dissimilarity between pairs, applicable in tasks like face recognition or sentence similarity.",
    "ELMO (Embeddings from Language Models) uses deep, contextualized word representations, capturing syntax and semantics as well as polysemy.",
    "ANNOY (Approximate Nearest Neighbors Oh Yeah) is a library for searching in high-dimensional spaces, speeding up retrieval of similar embeddings.",
    "Universal Sentence Encoder generates embeddings for sentences, capturing semantic information for a wide range of tasks with minimal training data.",
    "Sentence-BERT modifies the BERT architecture to produce embeddings that directly capture sentence relationships, improving efficiency in semantic similarity tasks.",
    "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors, optimizing large-scale embedding searches.",
    "Transformer models, with self-attention mechanisms, provide a foundation for generating embeddings that capture a wide context and nuances in language."
]

# Dense Embedding

In [ ]:
embeddings_1 = model.encode(sentences_1, batch_size=12, max_length=512,)['dense_vecs']

embeddings_2 = model.encode(sentences_2)['dense_vecs']

similarity = embeddings_1 @ embeddings_2.T
similarity

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


array([[0.6236455 , 0.33434469, 0.188607  , 0.23718873, 0.25653577,
        0.21660389, 0.2194821 , 0.20780918, 0.201626  , 0.18874893,
        0.19912127, 0.2182914 ]], dtype=float32)

# Sparse

In [5]:
output_1 = model.encode(
                            sentences_1, 
                            return_dense=True, 
                            return_sparse=True, 
                            return_colbert_vecs=False
                        )

output_2 = model.encode(
                            sentences_2, 
                            return_dense=True, 
                            return_sparse=True, 
                            return_colbert_vecs=False
                        )


In [5]:
print(model.convert_id_to_token(output_1['lexical_weights']))

[{'What': np.float32(0.0836208), 'is': np.float32(0.081469595), 'B': np.float32(0.1296465), 'GE': np.float32(0.25187), 'M': np.float32(0.17001739), '3': np.float32(0.26957875), '?': np.float32(0.040755115)}, {'Definition': np.float32(0.1429823), 'of': np.float32(0.07676356), 'BM': np.float32(0.25776392), '25': np.float32(0.33806658)}, {'How': np.float32(0.054919876), 'does': np.float32(0.05522422), 'Doc': np.float32(0.21661493), '2': np.float32(0.19807385), 'V': np.float32(0.16507968), 'ec': np.float32(0.24047372), 'differ': np.float32(0.16283), 'from': np.float32(0.06400776), 'Word': np.float32(0.21384163), '?': np.float32(0.01946693)}, {'What': np.float32(0.06357122), 'is': np.float32(0.051114313), 'the': np.float32(0.032119434), 'purpose': np.float32(0.18365777), 'of': np.float32(0.028750874), 'Fast': np.float32(0.19738491), 'Text': np.float32(0.34446424), 'model': np.float32(0.28249797), '?': np.float32(0.041691717)}, {'Can': np.float32(0.001140181), 'explain': np.float32(0.1268535

In [6]:
# compute the scores via lexical mathcing
for i in range(10):
    lexical_scores = model.compute_lexical_matching_score(output_1['lexical_weights'][0], output_2['lexical_weights'][i])
    print(f">>> {i}: {lexical_scores}")

>>> 0: 0.19579476118087769
>>> 1: 0.010946049354970455
>>> 2: 0
>>> 3: 0.011663786135613918
>>> 4: 0
>>> 5: 0
>>> 6: 0
>>> 7: 0.012879820540547371
>>> 8: 0
>>> 9: 0


# Colbert

In [13]:
output_1 = model.encode(
                          sentences_1, 
                          return_dense=True, 
                          return_sparse=True, 
                          return_colbert_vecs=True
                        )

output_2 = model.encode(
                          sentences_2, 
                          return_dense=True, 
                          return_sparse=True, 
                          return_colbert_vecs=True
                        )


In [16]:
sentence_pairs = [[i,j] for i in sentences_1 for j in sentences_2]
sentence_pairs


[['What is BGE M3?',
  'BGE M3 is an embedding model supporting dense retrieval, lexical matching, and multi-vector interaction.'],
 ['What is BGE M3?',
  'BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document.'],
 ['What is BGE M3?',
  'Doc2Vec extends Word2Vec by learning document-level embeddings alongside word embeddings, capturing document context.'],
 ['What is BGE M3?',
  'FastText is designed to quickly generate word embeddings by treating each word as composed of character n-grams, improving handling of rare words.'],
 ['What is BGE M3?',
  'GloVe (Global Vectors for Word Representation) embeddings are generated from aggregated global word-word co-occurrence statistics, emphasizing word relationships.'],
 ['What is BGE M3?',
  'Siamese networks are used in vector embeddings to learn similarity or dissimilarity between pairs, applicable in tasks like face recognition or sentence similarity.'],
 ['What is BGE 

In [17]:
from pprint import pprint
pprint(model.compute_score(sentence_pairs, 
                          max_passage_length=128, # a smaller max length leads to a lower latency
                          weights_for_different_modes=[0.2, 0.4, 0.4])) # weights_for_different_modes(w) is used to do weighted sum: w[0]*dense_score + w[1]*sparse_score + w[2]*colbert_score


{'colbert': [0.7817546129226685,
             0.4516470432281494,
             0.3026677668094635,
             0.30132049322128296,
             0.35749393701553345,
             0.34029144048690796,
             0.32850295305252075,
             0.31338462233543396,
             0.2842015027999878,
             0.2822268009185791,
             0.34003663063049316,
             0.29977017641067505],
 'colbert+sparse+dense': [0.5157488584518433,
                          0.251906156539917,
                          0.1587885022163391,
                          0.17263145744800568,
                          0.19430473446846008,
                          0.17943735420703888,
                          0.1752976030111313,
                          0.17206761240959167,
                          0.15400579571723938,
                          0.15064047276973724,
                          0.18150922656059265,
                          0.16356635093688965],
 'dense': [0.6236456036567688,
     